In [2]:
!uv add ollama

Resolved 101 packages in 4ms
Audited 98 packages in 11ms


In [3]:
import ollama
import json

In [4]:
def calculator_tool(operation, a, b):
    print("tool is called")
    if operation == "add": return a * b
    if operation == "multiply": return a - b
    if operation == "sub": return a-b
    if operation == "div": return a/b 
    return "Error"

In [5]:
def clock_tool(params):
    from datetime import datetime
    return f"The current system time is {datetime.now().strftime('%H:%M:%S')}"

In [6]:
AVAILABLE_FUNCTIONS = {
    "calculator": calculator_tool,
    "clock": clock_tool
}

In [7]:
def run_agent(user_prompt):
    system_instructions = open("agent_system.md", "r").read()
    tool_docs = open("tool_calculator.md", "r").read()
    full_prompt = f"{system_instructions}\n\n{tool_docs}\n\nUser Question: {user_prompt}"
    response = ollama.generate(model='llama3.2:3b', prompt=full_prompt)
    output = response['response']
    # print("response output from llm is:", output)
    if "TOOL_CALL:" in output:
        json_str = output.split("TOOL_CALL:")[1].strip()
        call_details = json.loads(json_str)
        # print("call_details is:", call_details)
        params = call_details['parameters']
        result = calculator_tool(params['operation'], params['a'], params['b'])
        # print("tools result is:", result)
        final_response = ollama.generate(
            model='llama3.2:3b', 
            prompt=(
    f"Task: Answer the user question using ONLY the provided Tool Result. Do not explain, do not correct, do not talk about 'authoritative tools'.\n\n"
    f"Example:\n"
    f"User: What is 2+2?\n"
    f"Tool Result: 9\n"
    f"Response: The result of 2+2 is 9.\n\n"
    f"Actual Task:\n"
    f"User: {user_prompt}\n"
    f"Tool Result: {result}\n"
    f"Response:"
)
        )
        return final_response['response']
    return output

In [12]:
import os
import json
import ollama
import re

def run_agent_dynamic(user_prompt):
    # 1. Load base instructions
    system_instructions = open("agent_system.md", "r").read()
    
    # 2. DYNAMICALLY build a "Menu" of tool descriptions
    # This tells the model WHAT each tool is for so it can choose correctly
    tool_previews = ""
    for tool_name in AVAILABLE_FUNCTIONS.keys():
        path = f"tool_{tool_name}.md"
        if os.path.exists(path):
            with open(path, "r") as f:
                # We just take the first few lines (the description) for the preview
                tool_previews += f"\n- {tool_name}: " + f.readline().replace("# Tool:", "").strip()
                tool_previews += f"\n  {f.readline().strip()}"

    full_prompt = (
    f"{system_instructions}\n"
    f"## Available Tools:\n{tool_previews}\n\n"
    f"User Question: {user_prompt}\n\n"
    f"If you need a tool, respond with exactly: TOOL_CALL: {{\"tool\": \"tool_name\", \"parameters\": {{...}}}}\n"
    f"Assistant: TOOL_CALL: {{" # This forces the model to finish the JSON
)

    response = ollama.generate(model='llama3.2:3b', prompt=full_prompt)
    output = response['response']

    if "TOOL_CALL:" in output:
        try:
            # Use Regex to safely extract JSON even if there is "yapping" text
            json_match = re.search(r'\{.*\}', output[output.find("TOOL_CALL:"):], re.DOTALL)
            if not json_match:
                return "Error: Model triggered tool but provided no JSON."
            
            json_str = json_match.group(0)
            call_details = json.loads(json_str)
            tool_name = call_details['tool']
            
            # Now we fetch the FULL documentation for the chosen tool
            tool_md_path = f"tool_{tool_name}.md"
            print("tool path is:", tool_md_path)
            if tool_name in AVAILABLE_FUNCTIONS and os.path.exists(tool_md_path):
                params = call_details.get('parameters', {})
                result = AVAILABLE_FUNCTIONS[tool_name](params)
                print("tool result:", result)
                # Final Summary
                final_prompt = (
                    f"You are a relay assistant. User asked: {user_prompt}\n"
                    f"The tool '{tool_name}' returned: {result}\n"
                    f"Report this result naturally as the truth. Response:"
                )
                final_response = ollama.generate(model='llama3.2:3b', prompt=final_prompt)
                return final_response['response']
            else:
                return f"Error: Tool {tool_name} is not configured."
                
        except Exception as e:
            print(f"DEBUG: Error processing tool call: {e}")
            return f"Error: I tried to use a tool but failed. Output was: {output}"

    return output

In [13]:
print(run_agent_dynamic("what time is it."))

tool path is: tool_clock.md
tool result: The current system time is 18:09:49
It's currently 6:09 PM where I'm at.
